In [1]:
import torch.nn as nn
from girth import grm_mml
from torch.utils.data import dataset, DataLoader


In [2]:
import torch
from torch import nn
from torch.nn import functional as F


class VSOSPred(nn.Module):
    def __init__(self, num_usrs: int, num_tasks: int):
        super().__init__()
        self.global_strong = nn.Embedding(num_usrs, 1)
        self.performance = nn.Embedding(num_usrs * num_tasks, 1)
        nn.init.zeros_(self.global_strong.weight)
        nn.init.zeros_(self.performance.weight)
        self.num_tasks = num_tasks
    def get_strong(self, usr_ids, task_ids):
        global_strength = self.global_strong(usr_ids).squeeze(-1)
        joint_ids = usr_ids * self.num_tasks + task_ids
        local = self.performance(joint_ids).squeeze(-1)
        return global_strength + local
    def forward(self, usr_a_ids, usr_b_ids, task_ids):
        strength_a = self.get_strong(usr_a_ids, task_ids)
        strength_b = self.get_strong(usr_b_ids, task_ids)
        return {"logits": strength_a - strength_b, "strength_a": strength_a, "strength_b": strength_b}
    def regularization_loss(self, global_weight, offset_weight):
        global_penalty = self.global_strong.weight.pow(2).mean()
        offset_penalty = self.performance.weight.pow(2).mean()
        return (global_weight * global_penalty + offset_weight * offset_penalty)

In [14]:
from torch.utils.data import Dataset
import numpy as np
import pandas as pd
from dataset import VSOSet
df = pd.read_csv("learn_data.csv")
l = df['name'].values.tolist()
m = {b: a
     for a, b in enumerate(l)}
df.name = df.name.map(m)
df = df.drop(columns=['Unnamed: 0'])
dataset=VSOSet(df)
dataloader = DataLoader(dataset, shuffle=True, batch_size=64)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 475 entries, 0 to 474
Data columns (total 46 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           475 non-null    int64  
 1   total1         16 non-null     float64
 2   result1        16 non-null     float64
 3   result2        15 non-null     float64
 4   total3         198 non-null    float64
 5   grade2         198 non-null    float64
 6   stage          198 non-null    float64
 7   A              198 non-null    float64
 8   B              198 non-null    float64
 9   C              198 non-null    float64
 10  D              198 non-null    float64
 11  E              198 non-null    float64
 12  F              198 non-null    float64
 13  result3        8 non-null      float64
 14  grade3         8 non-null      float64
 15  score4         98 non-null     float64
 16  grade4         284 non-null    float64
 17  A1_x           284 non-null    float64
 18  B1_x           284 no

In [ ]:
import pandas as pd
df = pd.read_csv('learn_data.csv')
df.rename(columns={'Unnamed: 0': 'id'}, inplace=True)
name = df.name.values
df.drop(columns=['name'], inplace=True)
#df = df.drop(columns=[''])
df

In [ ]:
num_usrs = 500

In [ ]:
model = VSOSpred(
    num_usrs=480,
    num_olimps=7,
)
optim = torch.optim.AdamW(model.parameters())
for batch in dataloader:
    out = model(
        athlete_a_ids=batch["athlete_a"],
        athlete_b_ids=batch["athlete_b"],
        competition_ids=batch["competition_id"],
    )
    pairwise_loss = F.binary_cross_entropy_with_logits(
        out["logits"],
        batch["target"].float(),
    )
    loss = (
        pairwise_loss
        + model.regularization_loss(1e-3, 1e-1)
    )
    loss.backward()
    optim.step()
    optim.zero_grad()